<a href="https://colab.research.google.com/github/SirenSapre/Dashboard-API_Claude_Cost_TGR/blob/main/AI_Agents_with_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TASK 1. ENVIRONMENT SETUP & BUILD AN AI AGENT WITH NO MEMORY

In [1]:
!pip install openai-agents==0.10.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.0/413.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 7.4 MB/s eta 0:00:00


In [5]:
import os
from openai import OpenAI
from IPython.display import display, Markdown

from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Create an instance of the OpenAI client
openai_client = OpenAI(api_key = OPENAI_API_KEY)
print("OpenAI client configured.")

OpenAI client configured.


In [ ]:
# Load environment variables and configure client
# from dotenv import load_dotenv
# load_dotenv()
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [6]:
# Import required classes from the agents module
# Agent: Represents an AI agent with a specific role and instructions
# Runner: Executes the agent and handles interactions
# SQLiteSession: Provides persistent memory storage for the agent using SQLite

from agents import Agent, Runner, SQLiteSession
from IPython.display import Markdown, display

def print_markdown(text: str):
    display(Markdown(text))


# Define the role and instructions for the AI agent
finance_advisor_instructions = """
Context:
You are a friendly personal finance advisor who helps users track spending and make better budgeting decisions.

Instructions:
- Every time the user mentions spending, extract:
  - amount
  - currency if given
  - category (groceries, transport, eating out, rent, entertainment, etc.)
- Use conversation history (memory) to keep:
  - a running total of all expenses
  - totals per category
- When the user asks for advice or a summary:
  - calculate approximate totals from the conversation so far
  - identify 1–3 categories where they are spending the most
  - suggest specific, practical ways to reduce spending next week.
- Be concise, factual, and non-judgmental and don't hallucinate.

Output:
- When logging expenses, briefly confirm what you recorded.
- When giving advice or summaries, use Markdown with:
  - a short summary sentence
  - a bullet list of key numbers (totals, top categories)
  - 2–3 concrete suggestions to improve their budget.
"""

# Create an instance of the Agent
finance_advisor_agent = Agent(
    name = "Finance Advisor",
    instructions = finance_advisor_instructions,
    model = "gpt-5-mini")


# TASK 2. RUN AN AI AGENT WITH NO MEMORY

In [7]:
# Example: An AI Agent with NO memory

# Let's give our first question to the AI agent
user_input1 = "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week."
print(f"You: {user_input1}")

# Run the agent with the first question (no memory means each query is independent)
response1 = await Runner.run(
    starting_agent = finance_advisor_agent,
    input = user_input1)   # No memory enabled

print(f"🤖 Agent:\n{response1.final_output}")


You: I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week.


ERROR:openai.agents:Error getting response: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}. (request_id: req_7da5e685b366414eae44d9c9cca6cdb1)
ERROR:openai.agents:Error getting response; filtered.input=[{'content': 'I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week.', 'role': 'user'}]


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
# Second question — depends on previous context
# Follow-up question that refers to the previous answer
user_input2 = ("My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?")

# Display the follow-up question
print(f"You: {user_input2}")

# Run the agent again — since there’s no memory, it does not recall the first question/answer
response2 = await Runner.run(
    starting_agent = finance_advisor_agent,
    input = user_input2)

# Display the agent’s response (will fail to connect it to the first question)
print(f"🤖 Agent:\n{response2.final_output}")


# TASK 3. ADD MEMORY TO THE AGENT USING SQLITESESSION

In [ ]:
# Create a session instance
# SQLite-based implementation of session storage.
# This implementation stores conversation history in a SQLite database.
session = SQLiteSession("conversation")


# Let's give our first question to the AI agent
user_input1 = "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week."

# Display the user’s question in Markdown format
print(f"You: {user_input1}")

# Run the agent with the first question (no memory means each query is independent)
response1 = await Runner.run(
    starting_agent = finance_advisor_agent,
    input = user_input1,
    session = session)   # memory enabled

print(f"🤖 Agent:\n{response1.final_output}")


In [ ]:
# Follow-up question that refers to the previous answer
user_input2 = ("My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?")

# Display the follow-up question
print(f"You: {user_input2}")

# Run the agent again — since there’s no memory, it does not recall the first question/answer
response2 = await Runner.run(
    starting_agent = finance_advisor_agent,
    input = user_input2,
    session = session)   # <-- same session, so it remembers past expenses

# Display the agent’s response (will fail to connect it to the first question)
print(f"🤖 Agent:\n{response2.final_output}")


**PRACTICE OPPORTUNITY:**  
- **Using OpenAI Agents SDK, create a new AI agent named `Wellness Coach` that always suggests one simple weekly health habit.**  
    - **1. Write the agent’s instructions; tell it to give exactly one small, realistic habit per week and avoid repeating previous habits.**  
    - **2. Use the latest `gpt-5-mini` model.**  
    - **3. Save the conversation history with `SQLiteSession` so it remembers the user’s name, lifestyle, and previously suggested habits.**  
    - **4. Ask the agent:**  
      **`My name is Tamara. I work long hours at a desk and feel low energy in the afternoon. Suggest one simple health habit I can try next week.`**  
    - **5. Display the agent’s answer.**  
    - **6. Then ask a follow-up question (same session) asking for a NEW habit that builds on the user’s progress and does not repeat the previous suggestion:**  
      **`Assume I successfully followed your first habit for one week. Suggest a NEW habit for the following week that builds on my progress (do not repeat the previous habit).`**  
    - **Hint: Use `Runner.run()` to send both messages to the agent and print the replies.**


# PRACTICE OPPORTUNITY SOLUTIONS

**PRACTICE OPPORTUNITY SOLUTION:**  
- **Using OpenAI Agents SDK, create a new AI agent named `Wellness Coach` that always suggests one simple weekly health habit.**  
    - **1. Write the agent’s instructions; tell it to give exactly one small, realistic habit per week and avoid repeating previous habits.**  
    - **2. Use the latest `gpt-5-mini` model.**  
    - **3. Save the conversation history with `SQLiteSession` so it remembers the user’s name, lifestyle, and previously suggested habits.**  
    - **4. Ask the agent:**  
      **`My name is Tamara. I work long hours at a desk and feel low energy in the afternoon. Suggest one simple health habit I can try next week.`**  
    - **5. Display the agent’s answer.**  
    - **6. Then ask a follow-up question (same session) asking for a NEW habit that builds on the user’s progress and does not repeat the previous suggestion:**  
      **`Assume I successfully followed your first habit for one week. Suggest a NEW habit for the following week that builds on my progress (do not repeat the previous habit).`**  
    - **Hint: Use `Runner.run()` to send both messages to the agent and print the replies.**


In [ ]:
from agents import Agent, Runner, SQLiteSession

wellness_coach_instructions = """
You are a supportive health and wellness coach called Wellness Coach.
Your job is to help busy professionals build better habits one week at a time.

Rules:
- Always suggest exactly ONE simple, realistic habit per reply.
- Make the habit small and specific (e.g., "10-minute walk after lunch on weekdays").
- Give a brief explanation of why this habit helps.
- If the user talks to you again in the same session, remember the habits
  you already suggested and DO NOT repeat them. Suggest something new that
  builds on their progress.
- Keep the tone encouraging and non-judgmental.
Remember the user's name, work style, and energy concerns across the session.
"""


In [ ]:
# Define agent
wellness_coach_agent = Agent(
    name = "Wellness Coach",
    instructions = wellness_coach_instructions,
    model = "gpt-5-mini"
)

# Create SQLite session so the agent can remember past suggestions
session = SQLiteSession("wellness_coach_session")

# First user request
first_msg = """
My name is Tamara. I work long hours at a desk and feel low energy in the afternoon.
Suggest one simple health habit I can try next week.
"""

resp1 = await Runner.run(
    starting_agent = wellness_coach_agent,
    input = first_msg,
    session = session)

print(f"Wellness Coach (Week 1 Habit):\n\n{resp1.final_output}")


In [ ]:
followup_msg = """
Assume I successfully followed your first habit for one week.
Suggest a NEW habit for the following week that builds on my progress
"""

resp2 = await Runner.run(
    starting_agent = wellness_coach_agent,
    input = followup_msg,
    session = session,
)

print_markdown(f"Wellness Coach (Week 2 Habit):\n\n{resp2.final_output}")


- **Would love to connect with everyone on LinkedIn: www.linkedin.com/in/dr-ryan-ahmed**